In [21]:
from __future__ import print_function, absolute_import
from gm.api import *
from gm.model.storage import context
from gm.model import DictLikeAlgoOrder
from gm.pb.account_pb2 import AlgoOrder
import numpy as np
import pandas as pd
from datetime import  datetime,timezone, timedelta
import datetime
import copy
import plotly.express as px
import plotly.graph_objects as go
#pd.set_option('display.max_rows', None) #show all rows
pd.set_option('display.float_format', '{:.2f}'.format)

#掘金量化
set_token('807b8ba88782d7343bfe3ad918f33f93a2610ee8')
account_id='28edb8c3-ed6d-11f0-9ac0-00163e022aa6'

In [22]:
#MACD (moving average convergence divergence)
#      =diff - dea
stock='SZSE.002104'
start='2025-06-01'
end='2026-02-12'

In [23]:
history_data = history(symbol=stock, frequency='1d', start_time=start,  end_time=end, fields='close,eob', adjust=ADJUST_PREV, df= True)
history_data['eob'] = history_data['eob'].dt.date  #优化日期格式

In [24]:
def EMA(dylan,n):
    ''' EMA(expotential moving average)
        EMA=close*alpha + EMA【-1】*（1-alpha）
        alpha=2/(n+1)
    '''
    ema=[0] * len(dylan)
    alpha=2/(n+1)
    ema[0]=dylan[0]
    for i in range(1,len(dylan)):
        ema[i]=alpha*dylan[i] + (1-alpha)*ema[i-1]
    return ema

In [25]:
history_data['EMA_12']=EMA(history_data['close'].tolist(),n=12)
history_data['EMA_26']=EMA(history_data['close'].tolist(),n=26)
history_data['diff']=history_data['EMA_12']-history_data['EMA_26']
history_data['dea'] = EMA(history_data['diff'], 9)
#vol=(diff - dea)*2
history_data['vol']=(history_data['diff'] - history_data['dea']) * 2
history_data

,close,eob,EMA_12,EMA_26,diff,dea,vol
0,9.74,2025-06-03,9.74,9.74,0.00,0.00,0.00
1,9.54,2025-06-04,9.71,9.73,-0.02,-0.00,-0.03
2,10.50,2025-06-05,9.83,9.78,0.05,0.01,0.08
3,10.95,2025-06-06,10.00,9.87,0.13,0.03,0.20
4,12.05,2025-06-09,10.32,10.03,0.29,0.08,0.41
...,...,...,...,...,...,...,...
170,18.34,2026-02-06,18.25,18.69,-0.44,-0.44,0.01
171,18.31,2026-02-09,18.26,18.66,-0.40,-0.44,0.07
172,18.09,2026-02-10,18.23,18.62,-0.39,-0.43,0.08
173,18.40,2026-02-11,18.26,18.60,-0.34,-0.41,0.13


In [ ]:
#作图
fig = px.line(history_data, x='eob', y=['diff', 'dea'], color_discrete_sequence=['white','yellow'])  
bar = px.bar(history_data, x='eob', y='vol')
colors = ['red' if v > 0 else 'green' for v in history_data['vol']]
bar = go.Figure(data=[
    go.Bar(x=history_data['eob'], y=history_data['vol'], marker_color=colors,name='volume')])
for trace in bar.data:
    fig.add_trace(trace)

In [27]:
fig.show()